# RSNA Knee — cache_336_raptor cache, part 3/3 (CPU)

| Experiment | Notebook | What this version does |
|---|---|---|
| **EXP-043** | `hlly34/rsna-knee-cache-raptor-p2` | Raptor-repro cache part 3/3: 80 slices (22/18/15/10/15) over 2-98%, 140 mm @336 px, 2-98 pct, no canonicalisation |

Builds studies `[2::3]` of `cache_336_raptor` with `configs/exp/raptor_repro.yaml` and packs them into tar shards for vast.ai (`CACHE_PACKS=... bash scripts/vast/get_data.sh`).

1. **Series per slot** — sag fluid 22, 2nd sag 18, cor fluid 15, 2nd cor 10, axial 15 slices.
2. **Slices** sorted by position, sampled evenly over 2–98% of each series; no left/right mirroring.
3. **Crop** 140 mm → 336 px (0.417 mm/px); per-series 2–98 percentile → uint8.

Pipeline code lives in the attached dataset `rsna-knee-code` (repo `src/`, `configs/`, `scripts/`); this notebook only orchestrates it. Results are registered with `scripts/track.py` into `experiments/registry.csv` using the id above.

In [ ]:
EXP = {"id": "EXP-043", "notebook": "hlly34/rsna-knee-cache-raptor-p2", "what": "Raptor-repro cache part 3/3: 80 slices (22/18/15/10/15) over 2-98%, 140 mm @336 px, 2-98 pct, no canonicalisation"}
print("EXPERIMENT", EXP["id"], "-", EXP["what"])
CFG, NAME, PART, PARTS = 'raptor_repro', 'cache_336_raptor', 2, 3
import glob, json, os, subprocess, sys, time

def walk_find(pred, root="/kaggle/input", max_depth=7):
    """Depth-limited search of the attached inputs that never descends into the DICOM trees."""
    root = root.rstrip("/")
    for dirpath, dirnames, filenames in os.walk(root):
        depth = dirpath[len(root):].count(os.sep)
        dirnames[:] = [] if depth >= max_depth else sorted(d for d in dirnames if not d.endswith("_series"))
        hit = pred(dirpath, filenames)
        if hit:
            return hit
    return None

# Kaggle mounts inputs at /kaggle/input/{competitions,datasets,notebooks}/... - locate them instead of hardcoding
CODE = walk_find(lambda d, f: d if os.path.isfile(os.path.join(d, "src", "models", "inference.py")) else None)
COMP = walk_find(lambda d, f: d if "test.csv" in f and os.path.isdir(os.path.join(d, "test_series")) else None)
assert CODE and COMP, (CODE, COMP)
sys.path.insert(0, CODE)
print("code:", CODE, "| competition:", COMP)


In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "iterative-stratification",
                "pylibjpeg", "pylibjpeg-libjpeg", "pylibjpeg-openjpeg"], check=True)
DATA = "/kaggle/tmp/data"; os.makedirs(DATA, exist_ok=True)
if not os.path.exists(f"{DATA}/raw"):
    os.symlink(COMP, f"{DATA}/raw")
OUT = f"/kaggle/tmp/{NAME}"
ENV = dict(os.environ, DATA_DIR=DATA, PYTHONPATH=CODE)
t = time.time()
subprocess.run([sys.executable, f"{CODE}/scripts/preprocess.py", f"exp={CFG}", "preprocess.compress=true",
                f"preprocess.part={PART}", f"preprocess.parts={PARTS}",
                f"preprocess.num_workers={os.cpu_count()}", f"paths.cache_dir={OUT}"], env=ENV, check=True)
print(f"preprocess {(time.time() - t) / 60:.1f} min")

## Sanity checks, then pack

In [ ]:
import hashlib, tarfile
import numpy as np
files = sorted(f for f in os.listdir(OUT) if f.endswith(".npz"))
slices, cover = [], []
for f in files[::25]:
    z = np.load(os.path.join(OUT, f)); slices.append(z["image"].shape[0]); cover.append(float(z["valid"].mean()))
total = sum(os.path.getsize(os.path.join(OUT, f)) for f in files)
n_shards = max(1, round(total / 2**30))
manifest = {"name": NAME, "studies": len(files), "bytes": total, "shards": []}
for k in range(n_shards):
    shard = f"{NAME}_p{PART}_{k:02d}.tar"
    with tarfile.open(f"/kaggle/working/{shard}", "w") as tar:
        for f in files[k::n_shards]:
            tar.add(os.path.join(OUT, f), arcname=f"{NAME}/{f}")
    for f in files[k::n_shards]:
        os.remove(os.path.join(OUT, f))
    h = hashlib.sha256()
    with open(f"/kaggle/working/{shard}", "rb") as fh:
        for block in iter(lambda: fh.read(1 << 24), b""):
            h.update(block)
    manifest["shards"].append({"file": shard, "sha256": h.hexdigest(), "bytes": os.path.getsize(f"/kaggle/working/{shard}")})
json.dump(manifest, open("/kaggle/working/cache_manifest.json", "w"), indent=1)
EXP.update(studies=len(files), size_gib=round(total / 2**30, 2), shards=n_shards, part=PART, parts=PARTS,
           slices_per_study_mean=round(float(np.mean(slices)), 1), slices_per_study_max=int(np.max(slices)),
           slot_coverage=round(float(np.mean(cover)), 4))
json.dump(EXP, open("/kaggle/working/experiment.json", "w"), indent=1)
print(EXP)